In [1]:
# ============================================
# TASK 1: DATA PIPELINE DEVELOPMENT
# ============================================

# Step 1: Import required libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Step 2: Load customer churn dataset

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

# Display first 5 rows
print("Dataset Loaded Successfully!\n")

print(df.head())

Dataset Loaded Successfully!

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Stream

In [3]:
# Step 3: Display dataset information

print("Dataset Shape:", df.shape)

print("\nColumn Names:\n")
print(df.columns)

print("\nData Types:\n")
print(df.dtypes)

print("\nMissing Values:\n")
print(df.isnull().sum())

Dataset Shape: (7043, 21)

Column Names:

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Data Types:

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Chu

In [4]:
# ============================================
# DATA CLEANING
# ============================================

# Step 4: Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove unnecessary column
df.drop(columns=['customerID'], errors='ignore', inplace=True)

# Convert TotalCharges column to numeric
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

# Fill missing values using median
df['TotalCharges'] = df['TotalCharges'].fillna(
    df['TotalCharges'].median()
)

print("Data Cleaning Completed Successfully!")

Data Cleaning Completed Successfully!


In [5]:
# Step 5: Convert target column into numeric format

df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

print("\nTarget Variable Encoded Successfully!\n")

print(df[['Churn']].head())


Target Variable Encoded Successfully!

   Churn
0      0
1      0
2      1
3      0
4      1


In [6]:
# Step 6: Separate input features and target variable

X = df.drop('Churn', axis=1)

y = df['Churn']

# Split dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Data Split Completed!")

Data Split Completed!


In [7]:
# Step 7: Identify numerical and categorical columns

num_cols = X.select_dtypes(
    include=['int64', 'float64']
).columns

cat_cols = X.select_dtypes(
    include=['object']
).columns

print("\nNumerical Columns:\n")
print(list(num_cols))

print("\nCategorical Columns:\n")
print(list(cat_cols))


Numerical Columns:

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Columns:

['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [8]:
# ============================================
# CREATE PIPELINES
# ============================================

# Step 8: Numerical pipeline

num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Categorical pipeline

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

# Combine pipelines

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("\nPipeline Created Successfully!")


Pipeline Created Successfully!


In [9]:
# ============================================
# APPLY PIPELINE
# ============================================

# Step 9: Fit and transform training data

X_train_transformed = preprocessor.fit_transform(
    X_train
)

# Transform testing data

X_test_transformed = preprocessor.transform(
    X_test
)

print("\nPipeline Applied Successfully!")

print("\nTransformed Training Shape:",
      X_train_transformed.shape)


Pipeline Applied Successfully!

Transformed Training Shape: (5634, 45)


In [10]:
# ============================================
# DISPLAY TRANSFORMED DATA
# ============================================

# Step 10: Get transformed feature names

num_features = list(num_cols)

cat_features = list(
    preprocessor.named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(cat_cols)
)

all_features = num_features + cat_features

# Convert transformed data into DataFrame

X_train_df = pd.DataFrame(
    X_train_transformed.toarray()
    if hasattr(X_train_transformed, "toarray")
    else X_train_transformed,
    columns=all_features
)

# Display transformed data

print("\nTransformed Dataset Preview:\n")

print(X_train_df.head())


Transformed Dataset Preview:

   SeniorCitizen    tenure  MonthlyCharges  TotalCharges  gender_Female  \
0      -0.437749 -0.465683       -0.000474     -0.421345            1.0   
1      -0.437749  0.885537        1.074754      1.255888            1.0   
2      -0.437749 -1.284605       -1.376499     -1.002151            0.0   
3      -0.437749 -1.161766        0.177346     -0.907292            0.0   
4      -0.437749 -1.325551       -0.098524     -0.394513            0.0   

   gender_Male  Partner_No  Partner_Yes  Dependents_No  Dependents_Yes  ...  \
0          0.0         1.0          0.0            0.0             1.0  ...   
1          0.0         1.0          0.0            1.0             0.0  ...   
2          1.0         0.0          1.0            1.0             0.0  ...   
3          1.0         1.0          0.0            1.0             0.0  ...   
4          1.0         1.0          0.0            0.0             1.0  ...   

   StreamingMovies_Yes  Contract_Month-to-m

In [11]:
# ============================================
# SAVE OUTPUTS
# ============================================

# Step 11: Save transformed dataset

X_train_df.to_csv(
    'transformed_data.csv',
    index=False
)

# Save preprocessing pipeline

joblib.dump(
    preprocessor,
    'churn_pipeline.pkl'
)

print("\nFiles Saved Successfully!")

print("\nSaved Files:")
print("- transformed_data.csv")
print("- churn_pipeline.pkl")


Files Saved Successfully!

Saved Files:
- transformed_data.csv
- churn_pipeline.pkl
